In [2]:
from datasets import load_dataset, Features, Value, Sequence, List as DsList, Json

features = Features({
    'messages': DsList(Json(decode=True)),
    'tools': DsList({'type': Value('string'), 'function': {'name': Value('string'), 'description': Value('string'), 'parameters': {'$schema': Value('string'), 'type': Value('string'), 'properties': Json(decode=True), 'required': DsList(Value('string'))}}}),
    'source': Value('string'),
    'model': Value('string'),
    'tokens': {'input': Value('int64'), 'output': Value('int64'), 'reasoning': Value('int64'), 'cache': {'read': Value('int64'), 'write': Value('int64')}},
    'cost': Value('float64'),
    '_reconstructed': {'system': Value('bool'), 'systemRole': Value('string'), 'tools': Value('int64'), 'sessionId': Value('string'), 'title': Value('string')},
})

dataset = load_dataset("nkasmanoff/opencode-sft", split="train", features=features)

In [2]:
import pandas as pd

In [ ]:
train_df = pd.DataFrame(dataset)


,messages,tools,source,_reconstructed
0,"[{'role': 'system', 'content': 'You are openco...","[{'type': 'function', 'function': {'name': 'ba...",opencode,"{'system': True, 'systemRole': 'build', 'tools..."
1,"[{'role': 'system', 'content': 'You are openco...","[{'type': 'function', 'function': {'name': 'ba...",opencode,"{'system': True, 'systemRole': 'build', 'tools..."
2,"[{'role': 'system', 'content': 'You are openco...","[{'type': 'function', 'function': {'name': 'ba...",opencode,"{'system': True, 'systemRole': 'build', 'tools..."
3,"[{'role': 'system', 'content': 'You are openco...","[{'type': 'function', 'function': {'name': 'ba...",opencode,"{'system': True, 'systemRole': 'build', 'tools..."
4,"[{'role': 'system', 'content': 'You are openco...","[{'type': 'function', 'function': {'name': 'gl...",opencode,"{'system': True, 'systemRole': 'build', 'tools..."


In [33]:
train_df['system_prompt'] = train_df['messages'].apply(lambda x: x[0]['content'])
train_df['system_prompt_start'] = train_df['system_prompt'].apply(lambda x: x[:400])
train_df['initial_message'] = train_df['messages'].apply(lambda x: x[1]['content'] if len(x) > 1 else None)
train_df['num_steps'] = train_df['messages'].apply(lambda x: len(x))

In [37]:
train_df.sort_values(by='num_steps', ascending=False)

,messages,tools,source,_reconstructed,system_prompt,initial_message,system_prompt_start,num_steps
731,"[{'role': 'system', 'content': 'You are openco...","[{'type': 'function', 'function': {'name': 'ba...",opencode,"{'system': True, 'systemRole': 'build', 'tools...","You are opencode, an interactive CLI tool that...",The user attached the following context. Use i...,"You are opencode, an interactive CLI tool that...",734
291,"[{'role': 'system', 'content': 'You are OpenCo...","[{'type': 'function', 'function': {'name': 'ba...",frontier,"{'system': True, 'systemRole': 'beast', 'tools...","You are OpenCode, the best coding agent on the...",Review the vscode git repo downloaded. First a...,"You are OpenCode, the best coding agent on the...",733
621,"[{'role': 'system', 'content': 'You are openco...","[{'type': 'function', 'function': {'name': 'ba...",frontier,"{'system': True, 'systemRole': 'build', 'tools...","You are opencode, an interactive CLI tool that...",take a look at this folder and repo inside. Su...,"You are opencode, an interactive CLI tool that...",725
909,"[{'role': 'system', 'content': 'You are openco...","[{'type': 'function', 'function': {'name': 'ba...",frontier,"{'system': True, 'systemRole': 'build', 'tools...","You are opencode, an interactive CLI tool that...","""The evalkit package (src/evalkit/) turns raw ...","You are opencode, an interactive CLI tool that...",379
523,"[{'role': 'system', 'content': 'You are OpenCo...","[{'type': 'function', 'function': {'name': 'ba...",frontier,"{'system': True, 'systemRole': 'beast', 'tools...","You are OpenCode, the best coding agent on the...","Please ssh into nkasmanoff@raspberrypi02w, I a...","You are OpenCode, the best coding agent on the...",369
...,...,...,...,...,...,...,...,...
392,"[{'role': 'system', 'content': 'You are openco...","[{'type': 'function', 'function': {'name': 'ba...",opencode,"{'system': True, 'systemRole': 'build', 'tools...","You are opencode, an interactive CLI tool that...",None,"You are opencode, an interactive CLI tool that...",1
393,"[{'role': 'system', 'content': 'You are openco...","[{'type': 'function', 'function': {'name': 'ba...",opencode,"{'system': True, 'systemRole': 'build', 'tools...","You are opencode, an interactive CLI tool that...",None,"You are opencode, an interactive CLI tool that...",1
394,"[{'role': 'system', 'content': 'You are openco...","[{'type': 'function', 'function': {'name': 'ba...",opencode,"{'system': True, 'systemRole': 'build', 'tools...","You are opencode, an interactive CLI tool that...",None,"You are opencode, an interactive CLI tool that...",1
395,"[{'role': 'system', 'content': 'You are openco...","[{'type': 'function', 'function': {'name': 'ba...",opencode,"{'system': True, 'systemRole': 'build', 'tools...","You are opencode, an interactive CLI tool that...",None,"You are opencode, an interactive CLI tool that...",1


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from collections import Counter

sns.set_theme(style="whitegrid")

In [ ]:
# Parse _reconstructed metadata
train_df['system_role'] = train_df['_reconstructed'].apply(
    lambda x: x.get('systemRole') if isinstance(x, dict) else None
)
train_df['num_tools_defined'] = train_df['_reconstructed'].apply(
    lambda x: x.get('tools') if isinstance(x, dict) else None
)

# Extract tool calls from assistant messages
def get_tool_calls(messages):
    calls = []
    for m in messages:
        if m['role'] == 'assistant' and 'tool_calls' in m and m['tool_calls']:
            for tc in m['tool_calls']:
                if tc.get('type') == 'function':
                    calls.append(tc['function']['name'])
    return calls

def get_has_reasoning(messages):
    return any(m.get('reasoning_content') for m in messages if m['role'] == 'assistant')

def get_user_message_len(messages):
    for m in messages:
        if m['role'] == 'user':
            return len(m['content'])
    return 0

train_df['tool_calls_list'] = train_df['messages'].apply(get_tool_calls)
train_df['num_tool_calls'] = train_df['tool_calls_list'].apply(len)
train_df['has_reasoning'] = train_df['messages'].apply(get_has_reasoning)
train_df['initial_user_msg_len'] = train_df['messages'].apply(get_user_message_len)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

steps = train_df['num_steps']
axes[0, 0].hist(steps[steps < 100], bins=50, edgecolor='white')
axes[0, 0].set_title('Conversation Steps (capped at 100)')
axes[0, 0].set_xlabel('num_steps')
axes[0, 0].set_ylabel('count')

axes[0, 1].hist(steps, bins=50, edgecolor='white', log=True)
axes[0, 1].set_title('Conversation Steps (log scale)')
axes[0, 1].set_xlabel('num_steps')
axes[0, 1].set_ylabel('count (log)')

source_counts = train_df['source'].value_counts()
axes[0, 2].bar(source_counts.index, source_counts.values, edgecolor='white')
axes[0, 2].set_title('Source Distribution')
axes[0, 2].set_xlabel('source')
axes[0, 2].tick_params(axis='x', rotation=45)

role_counts = train_df['system_role'].value_counts()
axes[1, 0].bar(role_counts.index, role_counts.values, edgecolor='white')
axes[1, 0].set_title('System Role Distribution')
axes[1, 0].set_xlabel('role')
axes[1, 0].tick_params(axis='x', rotation=45)

tc = train_df['num_tool_calls']
axes[1, 1].hist(tc[tc < 50], bins=50, edgecolor='white')
axes[1, 1].set_title('Tool Calls per Conversation (capped at 50)')
axes[1, 1].set_xlabel('num_tool_calls')
axes[1, 1].set_ylabel('count')

if train_df['num_tools_defined'].notna().any():
    tools_def = train_df['num_tools_defined'].value_counts().sort_index()
    axes[1, 2].bar(tools_def.index.astype(str), tools_def.values, edgecolor='white')
    axes[1, 2].set_title('Number of Tools Defined')
    axes[1, 2].set_xlabel('tool count')

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

sns.boxplot(data=train_df, x='source', y='num_steps', ax=axes[0])
axes[0].set_title('Steps by Source')
axes[0].tick_params(axis='x', rotation=45)

valid = train_df[train_df['system_role'].notna()]
sns.boxplot(data=valid, x='system_role', y='num_steps', ax=axes[1])
axes[1].set_title('Steps by System Role')
axes[1].tick_params(axis='x', rotation=45)

sns.boxplot(data=train_df, x='source', y='num_tool_calls', ax=axes[2])
axes[2].set_title('Tool Calls by Source')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

ulen = train_df['initial_user_msg_len']
ulen = ulen[ulen > 0]
axes[0].hist(ulen, bins=50, edgecolor='white')
axes[0].set_title('Initial User Message Length (chars)')
axes[0].set_xlabel('length')
axes[0].set_ylabel('count')

all_tools = [t for tools in train_df['tool_calls_list'] for t in tools]
tool_counts = Counter(all_tools).most_common(15)
if tool_counts:
    names, counts = zip(*tool_counts)
    axes[1].barh(list(reversed(names)), list(reversed(counts)), edgecolor='white')
    axes[1].set_title('Most Used Tools')
    axes[1].set_xlabel('count')

reasoning_by_source = train_df.groupby('source')['has_reasoning'].mean() * 100
axes[2].bar(reasoning_by_source.index, reasoning_by_source.values, edgecolor='white')
axes[2].set_title('% Conversations with Reasoning')
axes[2].set_xlabel('source')
axes[2].set_ylabel('percent')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## Catalog of traces from VALID_MODELS

Filter the dataset to traces produced by the target models, then cluster by what the user is asking.

In [ ]:
import re

VALID_MODELS = ["big-pickle", "anthropic/claude-opus-4.8"]


def extract_model(system_prompt):
    if not isinstance(system_prompt, str):
        return None
    m = re.search(r"exact model ID is\s+([^\s\n]+)", system_prompt)
    if m:
        return m.group(1).strip()
    m = re.search(r"powered by the model named\s+([^\s\n\.]+)", system_prompt)
    if m:
        return m.group(1).strip()
    return None


def normalize_model(name):
    if not isinstance(name, str):
        return None
    n = name.strip()
    for prefix in ("openrouter/", "anthropic/"):
        if n.startswith(prefix):
            n = n[len(prefix):]
    return n


valid_norm = {normalize_model(m) for m in VALID_MODELS}

train_df['model'] = train_df['system_prompt'].apply(extract_model)
train_df['model_norm'] = train_df['model'].apply(normalize_model)
train_df['is_valid_model'] = train_df['model_norm'].isin(valid_norm)

print('Model value counts:')
print(train_df['model'].value_counts(dropna=False))
print()
print(f"Traces matching VALID_MODELS: {train_df['is_valid_model'].sum()}")

In [ ]:
catalog_df = train_df[train_df['is_valid_model']].copy().reset_index(drop=True)
catalog_df = catalog_df[catalog_df['initial_message'].notna()].reset_index(drop=True)
catalog_df['user_request'] = catalog_df['initial_message'].astype(str).str.strip()

# Drop repeated/test prompts: an identical user request across traces is
# almost always a smoke/eval test (e.g. \"who are you\") rather than real work,
# so it doesn't belong in a catalog of what users ask. Keep only the first
# occurrence of each distinct request. (The pre-finetuning apply_filters() in
# dataset/build_dataset.py dedups full trajectories, not raw prompts.)
n_before = len(catalog_df)
dup_counts = catalog_df['user_request'].value_counts()
repeated = dup_counts[dup_counts > 1]
catalog_df = catalog_df.drop_duplicates(subset='user_request', keep='first').reset_index(drop=True)
print(f"Dropped {n_before - len(catalog_df)} repeated (likely test) prompts "
      f"across {len(repeated)} distinct duplicated requests")
print(f"Catalog size (valid model, has request, deduped): {len(catalog_df)}")
catalog_df[['model', 'source', 'num_steps', 'num_tool_calls', 'user_request']].head(10)

In [ ]:
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = embed_model.encode(
    catalog_df['user_request'].tolist(),
    show_progress_bar=True,
    normalize_embeddings=True,
)
print('Embeddings shape:', embeddings.shape)

In [ ]:
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer

n_clusters = min(8, max(2, len(catalog_df) // 10))
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
catalog_df['cluster'] = kmeans.fit_predict(embeddings)

# Top keywords per cluster (via TF-IDF) to label what users are asking
vectorizer = TfidfVectorizer(max_features=2000, stop_words='english', ngram_range=(1, 2))
tfidf = vectorizer.fit_transform(catalog_df['user_request'])
terms = np.array(vectorizer.get_feature_names_out())

cluster_labels = {}
for c in sorted(catalog_df['cluster'].unique()):
    mask = (catalog_df['cluster'] == c).values
    mean_tfidf = np.asarray(tfidf[mask].mean(axis=0)).ravel()
    top = terms[mean_tfidf.argsort()[::-1][:5]]
    cluster_labels[c] = ', '.join(top)
    print(f"Cluster {c} (n={mask.sum()}): {cluster_labels[c]}")

catalog_df['cluster_label'] = catalog_df['cluster'].map(cluster_labels)

In [ ]:
import umap
import textwrap
import plotly.express as px

reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, metric='cosine', random_state=42)
coords = reducer.fit_transform(embeddings)
catalog_df['x'] = coords[:, 0]
catalog_df['y'] = coords[:, 1]

catalog_df['cluster_str'] = catalog_df['cluster'].apply(lambda c: f"C{c}: {cluster_labels[c]}")
catalog_df['hover_request'] = catalog_df['user_request'].apply(
    lambda t: '<br>'.join(textwrap.wrap(t[:400], 80))
)

fig = px.scatter(
    catalog_df, x='x', y='y', color='cluster_str',
    hover_data={'x': False, 'y': False, 'cluster_str': False,
                'model': True, 'source': True, 'num_steps': True,
                'num_tool_calls': True, 'hover_request': True},
    labels={'cluster_str': 'What users ask', 'hover_request': 'request'},
    title=f'User Requests for VALID_MODELS ({len(catalog_df)} traces, {n_clusters} clusters)',
    width=1100, height=750,
)
fig.update_traces(marker=dict(size=8, opacity=0.75))
fig.update_layout(legend_title_text='What users ask', xaxis_title='UMAP-1', yaxis_title='UMAP-2')

out_path = 'valid_models_request_clusters.html'
fig.write_html(out_path)
print(f'Saved interactive plot to {out_path}')

# Inline render needs nbformat>=4.2.0; fall back to the saved HTML if missing.
try:
    fig.show()
except ValueError as e:
    print(f'Inline render unavailable ({e}). Open {out_path} in a browser.')

In [ ]:
# Representative user requests per cluster
for c in sorted(catalog_df['cluster'].unique()):
    sub = catalog_df[catalog_df['cluster'] == c]
    print(f"\n=== Cluster {c} (n={len(sub)}): {cluster_labels[c]} ===")
    for req in sub['user_request'].head(3):
        print(' -', req[:160].replace('\n', ' '))